In [1]:
# %%
# Cell flow (REPLACE: cells 0–3). Idempotent midlayer export (RAW MAP OUTPUTS).
# - Loads ONNX_IN
# - Finds logits src robustly
# - Finds layer2/layer3 endpoints by shape inference
# - Removes any prior midfeat head + any nodes producing "features"/"logits"/"layer2_map"/"layer3_map"
# - Exposes raw tensors:
#     layer2_map = (N,512,28,28)
#     layer3_map = (N,1024,14,14)
#     logits     = logits tensor
# - Saves ONNX_OUT
# - Verifies with ONNX Runtime

import onnx
from onnx import helper, TensorProto, shape_inference
import onnxruntime as ort
from pathlib import Path

# ----------------------------
# 0) Paths
# ----------------------------
ONNX_IN  = Path("/home/hschatzle/monte-carlo-selection/data/models/resnet50_tl_20250829.onnx")
ONNX_OUT = Path("/home/hschatzle/monte-carlo-selection/data/models/resnet50_tl_20250829_with_midmaps.onnx")

L2_MAP_OUT = "layer2_map"
L3_MAP_OUT = "layer3_map"
LOGITS_OUT = "logits"

# ----------------------------
# 1) Load + choose logits src robustly
# ----------------------------
model = onnx.load(str(ONNX_IN))

existing_outs = [o.name for o in model.graph.output]
print("Existing graph outputs:", existing_outs)

def pick_logits_src(names):
    if "output" in names:
        return "output"   # common original logits tensor
    if "logits" in names:
        return "logits"   # from prior surgery
    if len(names) == 0:
        raise RuntimeError("Model has zero graph outputs. Unexpected.")
    return names[0]        # fallback

logits_src = pick_logits_src(existing_outs)
print("Logits source tensor:", logits_src)

# ----------------------------
# 2) Infer shapes and locate layer2/layer3 endpoints (block2/block3 ends)
#    Expected for 224x224:
#      layer2 end: (N,512,28,28)
#      layer3 end: (N,1024,14,14)
# ----------------------------
m_inf = shape_inference.infer_shapes(model)

def vi_shape(vi):
    tt = vi.type.tensor_type
    if not tt.HasField("shape"):
        return None
    dims = []
    for d in tt.shape.dim:
        if d.dim_value > 0:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(d.dim_param)  # symbolic
        else:
            dims.append(None)
    return dims

all_vis = list(m_inf.graph.value_info) + list(m_inf.graph.input) + list(m_inf.graph.output)
name2vi = {vi.name: vi for vi in all_vis}

def find_tensor_by_4d_shape(channels: int, hw: int):
    cands = []
    for name, vi in name2vi.items():
        shp = vi_shape(vi)
        if not shp or len(shp) != 4:
            continue
        _, C, H, W = shp
        if C == channels and H == hw and W == hw:
            cands.append(name)
    return cands

layer2_cands = find_tensor_by_4d_shape(512, 28)
layer3_cands = find_tensor_by_4d_shape(1024, 14)

assert layer2_cands, "Could not find a (N,512,28,28) tensor for block2/layer2."
assert layer3_cands, "Could not find a (N,1024,14,14) tensor for block3/layer3."

layer2_src = layer2_cands[-1]
layer3_src = layer3_cands[-1]

print("Layer2 (block2) source tensor:", layer2_src)
print("Layer3 (block3) source tensor:", layer3_src)

# ----------------------------
# 3) Cleanup (idempotent):
#    - remove any prior midfeat nodes by node name (old GAP/concat head)
#    - remove any node that outputs prior exposed tensors (features/logits/layer2_map/layer3_map)
#    - reset graph outputs (we'll redefine)
# ----------------------------
midfeat_node_names = {
    "MidFeat_L2_GAP", "MidFeat_L3_GAP",
    "MidFeat_L2_Flatten", "MidFeat_L3_Flatten",
    "MidFeat_Concat",
    "ExposeMidFeats", "ExposeLogitsMid",
    # new map head names (idempotent)
    "ExposeLayer2Map", "ExposeLayer3Map", "ExposeLogitsMaps",
}

def outs_set(n):
    return set(list(n.output))

bad_out_tensors = {"features", "logits", L2_MAP_OUT, L3_MAP_OUT, LOGITS_OUT}

kept_nodes = []
for n in model.graph.node:
    if n.name in midfeat_node_names:
        continue
    if not outs_set(n).isdisjoint(bad_out_tensors):
        continue
    kept_nodes.append(n)

model.graph.ClearField("node")
model.graph.node.extend(kept_nodes)

model.graph.ClearField("output")

# ----------------------------
# 4) Expose raw maps + logits via Identity
# ----------------------------
nodes_to_add = [
    helper.make_node("Identity", inputs=[layer2_src], outputs=[L2_MAP_OUT], name="ExposeLayer2Map"),
    helper.make_node("Identity", inputs=[layer3_src], outputs=[L3_MAP_OUT], name="ExposeLayer3Map"),
    helper.make_node("Identity", inputs=[logits_src], outputs=[LOGITS_OUT], name="ExposeLogitsMaps"),
]
model.graph.node.extend(nodes_to_add)

# ----------------------------
# 5) Declare outputs with decent metadata (omit shape if unknown)
# ----------------------------
m_inf2 = shape_inference.infer_shapes(model)

def find_vi(name: str):
    for vi in list(m_inf2.graph.value_info) + list(m_inf2.graph.input) + list(m_inf2.graph.output):
        if vi.name == name:
            return vi
    return None

def make_output_vi(src_name: str, out_name: str):
    vi = find_vi(src_name)
    if vi is None:
        return helper.make_tensor_value_info(out_name, TensorProto.FLOAT, None)

    tt = vi.type.tensor_type
    elem = tt.elem_type

    # If any dim unknown/symbolic, omit full shape
    for d in tt.shape.dim:
        if d.dim_value <= 0:
            return helper.make_tensor_value_info(out_name, elem, None)

    shape = [int(d.dim_value) for d in tt.shape.dim]
    return helper.make_tensor_value_info(out_name, elem, shape)

model.graph.output.extend([
    make_output_vi(layer2_src, L2_MAP_OUT),
    make_output_vi(layer3_src, L3_MAP_OUT),
    make_output_vi(logits_src,  LOGITS_OUT),
])

# ----------------------------
# 6) Save
# ----------------------------
onnx.save(model, str(ONNX_OUT))
print("Saved:", ONNX_OUT)

# ----------------------------
# 7) Verify with ONNX Runtime
# ----------------------------
sess = ort.InferenceSession(str(ONNX_OUT), providers=["CPUExecutionProvider"])

print("\nORT outputs:")
for o in sess.get_outputs():
    print(" ", o.name, o.shape, o.type)

print("\nORT inputs:")
for i in sess.get_inputs():
    print(" ", i.name, i.shape, i.type)


Existing graph outputs: ['output']
Logits source tensor: output
Layer2 (block2) source tensor: /layer2/layer2.3/relu_2/Relu_output_0
Layer3 (block3) source tensor: /layer3/layer3.5/relu_2/Relu_output_0
Saved: /home/hschatzle/monte-carlo-selection/data/models/resnet50_tl_20250829_with_midmaps.onnx

ORT outputs:
  layer2_map ['batch_size', 512, 28, 28] tensor(float)
  layer3_map ['batch_size', 1024, 14, 14] tensor(float)
  logits ['batch_size', 54] tensor(float)

ORT inputs:
  input ['batch_size', 3, 224, 224] tensor(float)
